# 15 — AI Agent Tool Calling

## Goal
Demonstrate the principle of a tool-using agent: the model may request a Python tool, Python runs that tool, and the result is provided back to the model.

**What happens using this code?** This notebook shows explicit, inspectable routing in Python. It deliberately avoids giving the LLM unrestricted access to the computer or database.

In [ ]:
from pathlib import Path
import os, sys, json
from dotenv import load_dotenv
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from tools.assess_farming_risk import assess_farming_risk
from tools.search_agricultural_knowledge import search_agricultural_knowledge

# A simple explicit tool registry. Add only tools that are safe and needed.
TOOLS = {
    "assess_farming_risk": lambda args: assess_farming_risk(args.get("question", "")),
    "search_agricultural_knowledge": lambda args: search_agricultural_knowledge(args.get("query", ""), k=int(args.get("k", 3))),
}

question = "My rice plants are turning yellow. What should I check?"
# The Python controller chooses tools based on intent and data availability.
results = {"risk": TOOLS["assess_farming_risk"]({"question": question})}
results["evidence"] = TOOLS["search_agricultural_knowledge"]({"query": question, "k": 3})
print(json.dumps(results, ensure_ascii=False, indent=2, default=str))
print("\nNext step: send only these controlled tool outputs to Gemini for a grounded explanation.")

**Agent versus chatbot:** a tool-using agent performs an action (such as searching documents), checks the result, and then decides what to do next. For a reliable agriculture system, keep tool permissions narrow and validate tool outputs.